In [39]:
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

In [40]:
X,y = make_classification(n_samples=10000, n_features=10, n_informative=3)

In [41]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=28)

In [42]:
dt = DecisionTreeClassifier(random_state=28)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)

print('Decision Tree Accuracy ->',accuracy_score(y_test, y_pred))

Decision Tree Accuracy -> 0.9


# Bagging

In [43]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.5,
    bootstrap=True,
    random_state=28,
    n_jobs=-1
)

In [44]:
bag.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.5,
                  n_estimators=500, n_jobs=-1, random_state=28)

In [45]:
y_pred = bag.predict(X_test)

In [46]:
print(accuracy_score(y_test, y_pred))

0.9415


In [47]:
bag.estimators_samples_[0].shape

(4000,)

In [48]:
bag.estimators_features_[0].shape

(10,)

# Bagging using SVM

In [49]:
bag = BaggingClassifier(
    estimator=SVC(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    random_state=28,
    n_jobs=-1
)

In [50]:
bag.fit(X_train, y_train)

BaggingClassifier(estimator=SVC(), max_samples=0.25, n_estimators=500,
                  n_jobs=-1, random_state=28)

In [51]:
y_pred = bag.predict(X_test)

In [52]:
print(accuracy_score(y_test, y_pred))

0.895


# Pasting

In [53]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=False,
    verbose=1,
    n_jobs=-1
)

In [54]:
bag.fit(X_train, y_train)

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 out of   8 | elapsed:    2.4s remaining:    7.1s
[Parallel(n_jobs=8)]: Done   8 out of   8 | elapsed:    2.5s finished


BaggingClassifier(bootstrap=False, estimator=DecisionTreeClassifier(),
                  max_samples=0.25, n_estimators=500, n_jobs=-1, verbose=1)

In [55]:
y_pred = bag.predict(X_test)
print(accuracy_score(y_test, y_pred))

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


0.9385


[Parallel(n_jobs=8)]: Done   2 out of   8 | elapsed:    0.1s remaining:    0.2s
[Parallel(n_jobs=8)]: Done   8 out of   8 | elapsed:    0.1s finished


# Random Subspaces

In [56]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=1.0,
    max_features=0.5,
    bootstrap=False,
    bootstrap_features=True,
    random_state=28,
    n_jobs = -1
)

In [57]:
bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9195


# Random Patches

In [58]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_features=0.5,
    max_samples=0.25,
    bootstrap=True,
    bootstrap_features=True,
    random_state=28,
    n_jobs = -1
)

In [59]:
bag.fit(X_train, y_train)
bag.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9195


# OOB Score

In [60]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.5,
    bootstrap=True,
    oob_score=True,
    random_state=28,
    n_jobs = -1
)

In [61]:
bag.fit(X_train, y_train)
y_pred = bag.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9415


In [62]:
bag.oob_score_

0.93025

# Appling GridSearchCV

In [63]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

In [70]:
param = {
    'estimator': [DecisionTreeClassifier()],
    'n_estimators': [100,500,1000],
    'max_samples': [0.3,0.5,1.0],
    'bootstrap': [True,False],
    'max_features': [0.5,0.7,1.0]
}

In [72]:
search = GridSearchCV(BaggingClassifier(), param_grid=param, cv=5,n_jobs=-1, verbose=2)

In [73]:
search.fit(X_train, y_train)
print(search.best_params_)
print(search.best_score_)


Fitting 5 folds for each of 54 candidates, totalling 270 fits
[CV] END bootstrap=True, estimator=DecisionTreeClassifier(), max_features=0.5, max_samples=0.3, n_estimators=100; total time=   2.1s
[CV] END bootstrap=True, estimator=DecisionTreeClassifier(), max_features=0.5, max_samples=0.3, n_estimators=100; total time=   2.2s
[CV] END bootstrap=True, estimator=DecisionTreeClassifier(), max_features=0.5, max_samples=0.3, n_estimators=100; total time=   2.2s
[CV] END bootstrap=True, estimator=DecisionTreeClassifier(), max_features=0.5, max_samples=0.3, n_estimators=100; total time=   2.2s
[CV] END bootstrap=True, estimator=DecisionTreeClassifier(), max_features=0.5, max_samples=0.3, n_estimators=100; total time=   2.2s
[CV] END bootstrap=True, estimator=DecisionTreeClassifier(), max_features=0.5, max_samples=0.3, n_estimators=500; total time=  10.4s
[CV] END bootstrap=True, estimator=DecisionTreeClassifier(), max_features=0.5, max_samples=0.3, n_estimators=500; total time=  10.6s
[CV] EN

In [74]:
search.best_params_

{'bootstrap': True,
 'estimator': DecisionTreeClassifier(),
 'max_features': 1.0,
 'max_samples': 1.0,
 'n_estimators': 1000}

In [75]:
search.best_score_

np.float64(0.9332499999999999)

In [78]:
bc = BaggingClassifier(estimator=DecisionTreeClassifier(),max_features=1.0, max_samples=1.0,n_estimators=10000,bootstrap=True,n_jobs=-1,verbose=2)

In [79]:
bc.fit(X_train,y_train)

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


Building estimator 1 of 1250 for this parallel run (total 10000)...
Building estimator 1 of 1250 for this parallel run (total 10000)...
Building estimator 1 of 1250 for this parallel run (total 10000)...
Building estimator 1 of 1250 for this parallel run (total 10000)...
Building estimator 1 of 1250 for this parallel run (total 10000)...
Building estimator 1 of 1250 for this parallel run (total 10000)...
Building estimator 1 of 1250 for this parallel run (total 10000)...
Building estimator 2 of 1250 for this parallel run (total 10000)...
Building estimator 2 of 1250 for this parallel run (total 10000)...
Building estimator 2 of 1250 for this parallel run (total 10000)...
Building estimator 2 of 1250 for this parallel run (total 10000)...
Building estimator 2 of 1250 for this parallel run (total 10000)...
Building estimator 1 of 1250 for this parallel run (total 10000)...
Building estimator 2 of 1250 for this parallel run (total 10000)...
Building estimator 2 of 1250 for this parallel r

BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=10000,
                  n_jobs=-1, verbose=2)

In [80]:
y_pred = bc.predict(X_test)

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


In [81]:
print(accuracy_score(y_test, y_pred))

0.943
